In [1]:
!git clone https://github.com/ai360-project-school-j-lens/jacobian-lens-gpt2.git

Cloning into 'jacobian-lens-gpt2'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 66 (delta 5), reused 11 (delta 5), pack-reused 38 (from 1)
Receiving objects: 100% (66/66), 1.79 MiB | 6.63 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [2]:
import sys

sys.path.append('/kaggle/working/jacobian-lens-gpt2')

In [4]:
from gpt2 import GPT2LensModel

In [6]:
gpt = GPT2LensModel.from_pretrained()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [8]:
import json

dataset_name = "/kaggle/working/jacobian-lens-gpt2/data" + "/evaluations/lens-eval-order-ops.json"

dataset = json.load(open(dataset_name))

In [9]:
dataset['items'][0]

{'name': 'parens-add-mult',
 'prompt': '(2 + 3) * 4 = ',
 'target': '20',
 'intermediates': ['5', 'multiplication']}

In [43]:
hidden_states = gpt.forward(
    gpt.encode(dataset['items'][0]['prompt']),
    output_hidden_states=True,
).hidden_states

In [44]:
gpt.encode(dataset['items'][0]['prompt']).shape

torch.Size([1, 10])

In [45]:
hidden_states[0].shape

torch.Size([1, 10, 768])

In [48]:
gpt.unembed(hidden_states[0][0, 1, :]).argmax()

tensor(7)

In [35]:
vocab = gpt.tokenizer.get_vocab()
token_to_item = {
    vocab[i] : i
    for i in vocab
}

In [61]:
import pandas as pd
import torch
from IPython.display import display

MAX_LEN = 12  # обрезка длинных токенов

def show(tok_id):
    r = repr(gpt.tokenizer.decode(tok_id))
    return r if len(r) <= MAX_LEN else r[:MAX_LEN - 1] + "…"

for sample in dataset['items'][0:5]:
    prompt = sample['prompt'].rstrip()
    ids = gpt.encode(prompt)
    with torch.no_grad():
        hidden_states = gpt.forward(ids, output_hidden_states=True).hidden_states

    # номер позиции в названии, чтобы колонки были уникальными
    cols = [f"{j}: {show(t)}" for j, t in enumerate(ids[0].tolist())]
    rows = [f"L{i}" for i in range(len(hidden_states))]

    preds, probs = [], []
    for h in hidden_states:
        p, top = gpt.unembed(h[0]).float().softmax(dim=-1).max(dim=-1)
        preds.append([show(t) for t in top.tolist()])
        probs.append(p.tolist())

    df = pd.DataFrame(preds, index=rows, columns=cols)
    conf = pd.DataFrame(probs, index=rows, columns=cols)

    display(
        df.style
          .background_gradient(cmap="Blues", axis=None, gmap=conf, vmin=0, vmax=1)
          .set_caption(prompt)
          .set_table_styles([
              {"selector": "caption", "props": "font-size: 1.1em; font-weight: bold; text-align: left;"},
              {"selector": "td, th", "props": "font-family: monospace; text-align: center; padding: 4px 10px;"},
          ])
    )

,0: '<|endoftex…,1: '(',2: '2',3: ' +',4: ' 3',5: ')',6: ' *',7: ' 4',8: ' ='
L0,'<|endoftex…,'(','2',' +',' 3',')',' *',' mathemat',' ='
L1,'\n','the','nd',' +','rd','\n',' *','th','========'
L2,'\n','click','nd',' +','rd','\n',' *','th','========'
L3,"','",'click','nd','/-','rd',' –','ERROR','th','========'
L4,"','",'click','nd','/+','rd','\n','ERROR','th','========'
L5,"','",'click',')','/-','rd','\n','ERROR','th','========'
L6,"','",'click',')','/+','rd','\n','ERROR','th','========'
L7,"','",'Reuters',')','/-',')','\n','ERROR','½','========'
L8,"','",'Reuters',')',' 1',')',' [+','ERROR',' +','========'
L9,"','",'Reuters',')',' 1',')',' +','(',' +',' 0'


,0: '<|endoftex…,1: '(',2: '7',3: ' -',4: ' 3',5: ')',6: ' *',7: ' 2',8: ' =',9: ' 8',10: '.',11: ' (',12: '9',13: ' -',14: ' 4',15: ')',16: ' *',17: ' 3',18: ' ='
L0,'<|endoftex…,'(','7',' -',' 3',')',' *',' 2',' =',' mathemat',' challeng',' mathemat','9',' -',' mathemat',' challeng',' *',' challeng',' ='
L1,'\n','the','th',' -','rd','\n',' *','nd','========','th','\n','see','21',' -','th','\n',' *','rd','========'
L2,'\n','click','th',' -','rd','\n',' *','nd','========','90','5','...)','74',' -','th',' )',' *','rd','========'
L3,"','",'click','th',' -','rd',' –','ERROR','nd','========','90','5','...)','39',' -','th',' )','ERROR','rd','========'
L4,"','",'click',')',')','rd',' –','ERROR','nd','========','79','5','...)',')','_-',')',' )','ERROR','rd','========'
L5,"','",'click',')',' )','rd','\n','ERROR','nd','========','79','5','...)',')',"')""'",')','\n','ERROR','rd','========'
L6,"','",'click',')',"')""'",'rd','\n','ERROR','nd','========','th','5','...)',')','\\-',')',' Loading',' 2','rd','========'
L7,"','",'Reuters',')',"')""'",')','\n','ERROR','nd','========','th','5','...)',')',' 3',')',' *',' 2','rd',' 8'
L8,"','",'Reuters',')',"')""'",')','\n',' *','nd','========','.','5','1',')',' 3',')',' *',' 2',' =',' 8'
L9,"','",'Reuters',')',' )',')','\n',' *','nd','========','.','5','7',')',' 3',')',' *',' 2',' =',' 8'


,0: '<|endoftex…,1: '2',2: ' *',3: ' (',4: '3',5: ' +',6: ' 4',7: ')',8: ' ='
L0,'<|endoftex…,'2',' *',' mathemat','3',' +',' mathemat',')',' ='
L1,'\n','nd',' *','see','rd',' +','th','\n','========'
L2,'\n','nd',' *','see','rd',' +','th','\n','========'
L3,"','",'nd','ERROR','see','rd','/+','th',' |','========'
L4,"','",'nd','ERROR','...)','rd','/+',')',' |','========'
L5,"','",'nd','ERROR',' )','rd','/+',')','\n','========'
L6,"','",'nd','ERROR','?)','rd','/+',')','\n','========'
L7,"','",'nd','ERROR','1',')',')/',')',' +','========'
L8,"','",'nd','ERROR','1',')',')/',')',' =','========'
L9,"','",'nd',' *','1',')',' 1',')',' +',' 1'


,0: '<|endoftex…,1: '(',2: '9',3: ' -',4: ' 6',5: ')',6: ' *',7: ' 2',8: ' ='
L0,'<|endoftex…,'(','9',' -',' 6',')',' *',' 2',' ='
L1,'\n','the','21',' -','th','\n',' *','nd','========'
L2,'\n','click','79',' -','th',' )',' *','nd','========'
L3,"','",'click','79',' -','th',' Sep','ERROR','nd','========'
L4,"','",'click',')',"')""'",'th',' –','ERROR','nd','========'
L5,"','",'click',')',"')""'",'34','\n','ERROR','nd','========'
L6,"','",'click',')',"')""'",')','\n','ERROR','nd','========'
L7,"','",'Reuters',')',"')""'",')','\n','ERROR','nd','========'
L8,"','",'Reuters',')',"')""'",')','\n','ERROR','nd','========'
L9,"','",'Reuters',')',' 9',')','\n',' *','nd','========'


,0: '<|endoftex…,1: '2',2: ' +',3: ' 3',4: ' *',5: ' 4',6: ' ='
L0,'<|endoftex…,'2',' +',' 3',' *',' 4',' ='
L1,'\n','nd',' +','rd',' *','th','========'
L2,'\n','nd','++++++++++…,'rd',' *','th','========'
L3,"','",'nd','++++++++++…,'rd',' *','th','========'
L4,"','",'nd','-+-+-+-+','rd',' *','54','========'
L5,"','",'nd','-+-+-+-+','rd','ERROR','34','========'
L6,"','",'nd','++++++++++…,'rd','$',' +','========'
L7,"','",'nd','++++++++++…,'rd','ERROR',' +','========'
L8,"','",'nd','++++++++++…,' +',' +',' +','========'
L9,"','",'nd',' 1',' +',' 3',' +',' 1'


In [69]:
hidden_states = gpt.forward(ids, output_hidden_states=True)

In [73]:
prompt = "The Eifflel Tower is located in the city of"
ids = gpt.encode(prompt)
with torch.no_grad():
    res = gpt.forward(ids, output_hidden_states=True)
    hidden_states = [*res.hidden_states, res.last_hidden_state]

# номер позиции в названии, чтобы колонки были уникальными
cols = [f"{j}: {show(t)}" for j, t in enumerate(ids[0].tolist())]
rows = [f"L{i}" for i in range(len(hidden_states))]

preds, probs = [], []
for h in hidden_states:
    p, top = gpt.unembed(h[0]).float().softmax(dim=-1).max(dim=-1)
    preds.append([show(t) for t in top.tolist()])
    probs.append(p.tolist())

df = pd.DataFrame(preds, index=rows, columns=cols)
conf = pd.DataFrame(probs, index=rows, columns=cols)

display(
    df.style
      .background_gradient(cmap="Blues", axis=None, gmap=conf, vmin=0, vmax=1)
      .set_caption(prompt)
      .set_table_styles([
          {"selector": "caption", "props": "font-size: 1.1em; font-weight: bold; text-align: left;"},
          {"selector": "td, th", "props": "font-family: monospace; text-align: center; padding: 4px 10px;"},
      ])
)

,0: '<|endoftex…,1: 'The',2: ' E',3: 'iff',4: 'le',5: 'l',6: ' Tower',7: ' is',8: ' located',9: ' in',10: ' the',11: ' city',12: ' of'
L0,'<|endoftex…,'theless',' E','iff','le','l',' Tower',' destro',' located',' destro',' destro',' city',' destro'
L1,'\n','ories','ves','iff','le','azy',' Tower',' not',' located',' the',' same','scape',' the'
L2,'\n',' latest','Z','iculty','le','mer',' Tower',' now',' located',' order',' same','scape',' the'
L3,"','",' latest','Z','ness','ttes','azy',' Tower',' now',' located',' order',' same','scape',' the'
L4,"','",' latest','leg','ness','lla','azy',' Tower',' now',' located',' the',' same','scape',' the'
L5,"','",'resa','leg','y','lla','ott',' Tower',' also',' located',' front',' same','scape',' the'
L6,"','",'resa','FP','ler',' Club','ott',' Towers',' located',' located',' the',' same','scape',' England'
L7,"','",'resa','RC','ler',' Club','ott',' Tower',' located',' located',' the',' middle',' center',' South'
L8,"','",'resa','FP','ler',' Club','ée',' Tower',' located',' near',' downtown',' midst',' centre',' Ing'
L9,"','",'resa','Z','ert',' Club','ée',' Tower',' located',' near',' downtown',' midst',' centre',' Chicago'
